In [ ]:
"""
이동평균류(ITS 교통량 기반) 피처 사전 생성 스크립트

출력: output/features/speed_features.parquet
  columns: segment_key, timestamp, V_segment,
           speed_last_10min, speed_ma_30min, speed_ma_1h, speed_change_rate

핵심 결정: raw ITS 5분 통행속도 원본(링크 단위, 52GB/14억 행)은 필요 없다.
  이동평균은 "구간x방향 단위로 이미 길이가중평균까지 끝난 시계열"
  (output/segment_weighted_speed_final.parquet, segment_weighted_speed 생성
  스크립트 참고)을 rolling만 하면 되므로, link 단위 원본을 다시 읽을 필요가
  없다. 그 스크립트에서 이미 link -> segment_id x direction 가중평균 집계를
  끝내놓았기 때문에, 이번 작업은 그 결과 위에서 시계열 파생만 하면 된다.

왜 미리 만들어서 저장하는가 (is_bottleneck_slot과의 차이):
  이동평균/변화율은 그 시점 기준 "과거"만 보는 causal 피처라, 전체 기간에
  대해 한 번 계산해도 leakage가 생기지 않는다(Train/Val/Test를 나중에 어떻게
  나누든 각 row의 값은 바뀌지 않음). 반면 is_bottleneck_slot은 여러 날의
  통계(평균/표준편차)를 집계한 피처라 Train 컷오프가 바뀌면 값 자체가
  바뀌므로 그때그때 다시 계산해야 한다. 이 차이 때문에 이동평균류는
  미리 계산해서 parquet으로 캐싱해두는 쪽이 합리적이다(반복되는 XGBoost
  실험/튜닝마다 재계산할 필요가 없고, 학습 배치와 실시간 추론이 동일한
  로직을 공유하도록 함수화해두기도 쉬움).

시간 기준(time-based) rolling을 쓰는 이유:
  원본 5분 데이터에 결측으로 인한 불규칙한 간격(5, 10, 15, 20분 등)이 섞여
  있음을 EDA에서 확인했다. row 개수 기준 rolling(예: "최근 6개 row")을 쓰면
  결측 구간에서 실제 시간 폭과 안 맞는 왜곡이 생기므로, timestamp 기준
  시간 폭(window_size="10m"/"30m"/"1h")으로 rolling한다.
"""

import polars as pl
from pathlib import Path

INPUT_PATH = "./output/segment_weighted_speed_final.parquet"
OUTPUT_DIR = Path("./output/features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "speed_features.parquet"

In [ ]:
# ------------------------------------------------------------------
# 1. 데이터 로드 + segment_key 생성 + 정렬
# ------------------------------------------------------------------
# rolling_*_by는 파티션(segment_key) 내부가 timestamp 오름차순으로 정렬되어
# 있어야 정확하게 동작하므로, segment_key -> timestamp 순으로 반드시 정렬한다.

df = pl.read_parquet(INPUT_PATH)

df = df.with_columns(
    (pl.col("segment_id") + "_" + pl.col("direction")).alias("segment_key")
).select(["segment_key", "timestamp", "V_segment"])

df = df.sort(["segment_key", "timestamp"])

print(f"원본 shape: {df.shape}")
print(f"segment_key 수: {df['segment_key'].n_unique()}")
print(f"기간: {df['timestamp'].min()} ~ {df['timestamp'].max()}")

In [ ]:
# ------------------------------------------------------------------
# 2. 시간 기준 rolling으로 이동평균류 피처 생성
# ------------------------------------------------------------------
# speed_last_10min : t 시점 기준 직전 10분 실측 평균속도
# speed_ma_30min   : t 시점 기준 최근 30분 이동평균 (단기 변화 포착)
# speed_ma_1h      : t 시점 기준 최근 1시간 이동평균 (중기 흐름)
# speed_change_rate: speed_ma_30min - speed_ma_1h (악화 속도/가속도)
#
# closed="right": 윈도우가 현재 시점(t) 자신을 포함하고 그 이전만 본다
# (미래 데이터를 참조하지 않는 causal 계산 -> leakage 없음).
# .over("segment_key")로 구간x방향별로 독립적으로 rolling.

df = df.with_columns(
    [
        pl.col("V_segment")
        .rolling_mean_by("timestamp", window_size="10m", closed="right")
        .over("segment_key")
        .alias("speed_last_10min"),
        pl.col("V_segment")
        .rolling_mean_by("timestamp", window_size="30m", closed="right")
        .over("segment_key")
        .alias("speed_ma_30min"),
        pl.col("V_segment")
        .rolling_mean_by("timestamp", window_size="1h", closed="right")
        .over("segment_key")
        .alias("speed_ma_1h"),
    ]
).with_columns(
    (pl.col("speed_ma_30min") - pl.col("speed_ma_1h")).alias("speed_change_rate")
)

print(df.head(10))
print()
print("결측치 개수:")
print(df.null_count())

In [ ]:
# ------------------------------------------------------------------
# 3. 검증 + 저장
# ------------------------------------------------------------------
# 각 segment_key의 첫 관측치는 window 안에 자기 자신만 있으므로
# speed_last_10min == speed_ma_30min == speed_ma_1h == V_segment로 나오는 게
# 정상이다 (결측이 아니라 '아직 과거 데이터가 짧아서 자기 자신만 평균낸' 상태).

sample_key = df["segment_key"][0]
print(f"[{sample_key}] 첫 5개 행 (윈도우 초기 구간 확인용)")
print(df.filter(pl.col("segment_key") == sample_key).head(5))

final_cols = [
    "segment_key",
    "timestamp",
    "V_segment",
    "speed_last_10min",
    "speed_ma_30min",
    "speed_ma_1h",
    "speed_change_rate",
]
df.select(final_cols).write_parquet(OUTPUT_PATH)

print(f"\n저장 완료: {OUTPUT_PATH.resolve()}")
print(f"최종 shape: {df.select(final_cols).shape}")